::: {.callout-important}
## Main Idea
For a scalar loss, backpropagation is reverse-mode AD applied to the computation graph. Written as constraints, that reverse pass is the **discrete adjoint method**: a transposed linearized system solved backward.
:::

Three names get used for closely related things, and the mix of names can be confusing.

- *Reverse-mode AD*: the general technique. Sweep backward through a program to get how a chosen output responds to many inputs. This is the umbrella term.
- *Backpropagation*: the machine-learning name for reverse-mode AD applied to a scalar loss, returning gradients with respect to model parameters.
- *The adjoint method*: the same reverse sensitivity idea, usually written in the language of constrained optimization, inverse problems, and PDE-constrained models.

For the scalar objectives in this notebook, these viewpoints coincide: backprop is reverse-mode AD seeded at the loss, and the resulting backward sweep is the discrete adjoint of the computation being differentiated. Reverse-mode AD is more general than this setting, but this is the case that makes training and inversion practical.


In [1]:
import jax, jax.numpy as jnp, numpy as np, diffrax as dfx
jax.config.update("jax_enable_x64", True)
print("jax", jax.__version__, "| diffrax", dfx.__version__)

jax 0.10.2 | diffrax 0.7.2


## 1 · The problem: a gradient through a constraint

* A scalar loss $L(u,	\theta)\in\mathbb{R}$ depends on parameters $\theta\in\mathbb{R}^p$ through a state $u\in\mathbb{R}^n$.
* The state $u$ is not usually given by a closed-form formula. It is defined by a constraint:

$$g(u,\theta) = 0,\qquad g:\mathbb{R}^n\times\mathbb{R}^p\to\mathbb{R}^n \quad\Longrightarrow\quad u = u(\theta).$$

This shape appears in very different problems:

- *Glacier inversion*:
    - $g=0$ could be a discretized ice-flow equation or a coupled set of equations.
    - $\theta$ might represent parameters such as the basal friction coefficient $\beta$.
    - $L$ might measure mismatch between modeled and observed surface velocities.
- *Neural network*:
    - $g=0$ can represent the sequence of layers, after rewriting function composition as constraints.
    - $\theta$ is the collection of network parameters.
    - $L$ is the training loss.

::: {.callout-important}
We want $\dfrac{dL}{d\theta}\in\mathbb{R}^p$ without doing one expensive state-sensitivity solve per parameter.
:::


## 2 · Two ways to get the gradient

We want $dL/d\theta$, but $u$ is tied to $\theta$ by the constraint. Three steps:

**Step 1: chain rule.** $L$ depends on $\theta$ directly and through $u(\theta)$:

$$\frac{dL}{d\theta} = \frac{\partial L}{\partial\theta} + \frac{\partial L}{\partial u}\,\frac{du}{d\theta},
\qquad \frac{\partial L}{\partial u}\in\mathbb{R}^{1\times n},\quad \frac{du}{d\theta}\in\mathbb{R}^{n\times p}.$$

The hard part is $du/d\theta$: how the state responds to the parameters. This relationship is often defined implicitly via the constraint.

**Step 2: differentiate the constraint.** Since $g(u(\theta),\theta)=0$ for all $\theta$:

$$\frac{\partial g}{\partial u}\frac{du}{d\theta} + \frac{\partial g}{\partial\theta} = 0
\quad\Longrightarrow\quad
\frac{du}{d\theta} = -\Big(\frac{\partial g}{\partial u}\Big)^{-1}\frac{\partial g}{\partial\theta}.$$

**Step 3: substitute back.**

$$\frac{dL}{d\theta} = \frac{\partial L}{\partial\theta} - \frac{\partial L}{\partial u}\Big(\frac{\partial g}{\partial u}\Big)^{-1}\frac{\partial g}{\partial\theta}.$$

The gradient is explicit; all that remains is *how* to evaluate the middle product. The order of operations matters!

### Option 1: direct / tangent-linear (forward)

Build the sensitivity $du/d\theta$ (size $n\times p$) one column at a time. Column $j$ (how the state responds to parameter $\theta_j$) is the linear solve

$$\frac{\partial g}{\partial u}\,\Big(\frac{du}{d\theta}\Big)_{\!\cdot j} = -\Big(\frac{\partial g}{\partial\theta}\Big)_{\!\cdot j}.$$

- Same matrix $\partial g/\partial u$, a different right-hand side for each parameter, so $p$ solves, one per parameter.
- Plug the result into Step 1 to get $dL/d\theta$.
- This is the tangent-linear model, or *forward mode*:

> Pushes a perturbation of $\theta$ *forward* through the system. Efficient when the number of input directions is small.

### Option 2: adjoint / cotangent (reverse)

- Evaluate right to left: we never need $du/d\theta$ alone, only its product with $\partial L/\partial u$.
- Group the leading factors into the adjoint variable $\lambda\in\mathbb{R}^n$:

$$\lambda^{\top} := \frac{\partial L}{\partial u}\Big(\frac{\partial g}{\partial u}\Big)^{-1}
\quad\Longleftrightarrow\quad
\Big(\frac{\partial g}{\partial u}\Big)^{\!\top}\lambda = \Big(\frac{\partial L}{\partial u}\Big)^{\!\top}.$$

- That is the **adjoint equation**: one transposed solve, independent of the number of parameters $p$.
- Then
$$\frac{dL}{d\theta} = \frac{\partial L}{\partial\theta} - \lambda^{\top}\frac{\partial g}{\partial\theta}.$$

The final gradient still has $p$ entries, so accumulating or storing it still involves more work as the number of parameters grows. However, this involves only one expensive linear solve!

> Propagates a cotangent *backward*: this is *reverse mode*.

::: {.callout-important}
## The asymmetry that makes ML and inversion practical
For a scalar loss with many parameters, the direct method costs $O(p)$ state-sensitivity solves. The adjoint method costs one transposed solve, followed by local gradient accumulation. That is why we can fit models with millions of parameters.
:::

Same gradient, very different cost. Confirm on a small linear constraint $g(u,\theta)=Au-b(\theta)=0$:


In [2]:
# Constrained problem:  g(u, theta) = A u - b(theta) = 0  ->  u(theta) = A^{-1} b(theta).
# Loss L(u) = 1/2 ||u - u_target||^2.  DIRECT (one solve per parameter) vs ADJOINT (one solve).
n, p = 6, 5                                                     # state dim, number of parameters
Mn = jax.random.normal(jax.random.PRNGKey(0), (n, n))
A  = Mn @ Mn.T + n * jnp.eye(n)                                 # SPD system matrix
B  = jax.random.normal(jax.random.PRNGKey(1), (n, p))          # b(theta) = B @ theta
theta    = jax.random.normal(jax.random.PRNGKey(2), (p,))
u_target = jnp.ones(n)

def solve_u(theta): return jnp.linalg.solve(A, B @ theta)      # the forward solve, u(theta)
u    = solve_u(theta)
dLdu = u - u_target                                            # dL/du

# DIRECT / tangent-linear:  build du/dtheta column by column -> one linear solve PER PARAMETER
dudtheta    = jnp.stack([jnp.linalg.solve(A, B[:, j]) for j in range(p)], axis=1)   # p solves
grad_direct = dLdu @ dudtheta

# ADJOINT / cotangent:  ONE transposed solve, then dot products
lam          = jnp.linalg.solve(A.T, dLdu)                     # (dg/du)^T lambda = (dL/du)^T  <- 1 solve
grad_adjoint = lam @ B                                         # dL/dtheta = -lambda^T (dg/dtheta), dg/dtheta = -B

g_jax = jax.grad(lambda th: 0.5 * jnp.sum((solve_u(th) - u_target) ** 2))(theta)
print(f"parameters p = {p}")
print(f"  direct  : {p} linear solves   grad = {np.round(np.array(grad_direct), 5)}")
print(f"  adjoint : 1 linear solve      grad = {np.round(np.array(grad_adjoint), 5)}")
print(f"  adjoint matches direct: {bool(jnp.allclose(grad_direct, grad_adjoint))}"
      f"   |   matches jax.grad: {bool(jnp.allclose(grad_adjoint, g_jax))}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


parameters p = 5
  direct  : 5 linear solves   grad = [ 0.25687 -0.03022  0.0516  -0.32131  0.01129]
  adjoint : 1 linear solve      grad = [ 0.25687 -0.03022  0.0516  -0.32131  0.01129]
  adjoint matches direct: True   |   matches jax.grad: True


## 3 · The same equation everywhere: solvers and neural networks

- Nothing above required $g$ to be a PDE. It can represent any differentiable computation written as constraints.
- For example, we can compute the adjoint for a sequence of discrete steps:

### Example A: a time-stepping solver

Stack every time level into one big unknown $\mathbf{U}=(u_0,\dots,u_N)$, each $u_k\in\mathbb{R}^d$, and write one constraint per step (bold marks the stacked objects $\mathbf{U}$, $\boldsymbol{\lambda}$, versus the per-step $u_k$, $\lambda_k$):

$$g_0 = u_0 - u_{\text{init}},\qquad g_{k+1} = u_{k+1} - \Phi(u_k,\theta)\quad (k=0,\dots,N-1).$$

The adjoint of this *whole system* is §2's equation with $u\to\mathbf{U}$, one transposed linear solve for the big adjoint $\boldsymbol{\lambda}=(\lambda_0,\dots,\lambda_N)$:

$$\Big(\frac{\partial g}{\partial \mathbf{U}}\Big)^{\!\top}\boldsymbol{\lambda} = \Big(\frac{\partial L}{\partial \mathbf{U}}\Big)^{\!\top}.$$

We usually never build this full system. Its structure does the work for us. For an explicit one-step method, $\partial g/\partial \mathbf{U}$ is block lower-bidiagonal, so its transpose is block upper-bidiagonal:

$$\frac{\partial g}{\partial \mathbf{U}}=
\begin{pmatrix}
I & & & \\
-\Phi'_0 & I & & \\
& \ddots & \ddots & \\
& & -\Phi'_{N-1} & I
\end{pmatrix},
\qquad
\Big(\frac{\partial g}{\partial \mathbf{U}}\Big)^{\!\top}=
\begin{pmatrix}
I & -\Phi'^{\top}_0 & & \\
& I & \ddots & \\
& & \ddots & -\Phi'^{\top}_{N-1} \\
& & & I
\end{pmatrix}.$$

An upper-triangular system solves by back-substitution from the bottom up, so the one big solve decomposes into a per-step recurrence:

$$\lambda_N = \frac{\partial L}{\partial u_N},\qquad
\lambda_k = \frac{\partial L}{\partial u_k} + \Phi'^{\top}_k\,\lambda_{k+1}\ \ (k=N\!-\!1,\dots,0).$$

The adjoint of the entire trajectory is this cotangent carried backward, one step at a time. The structure of $\partial g/\partial \mathbf{U}$ turns a large transposed solve into a local backward sweep.

Each step does one local operation:

- $\Phi'^{\top}_k\lambda_{k+1}$ applies the step's *transpose-Jacobian* to the incoming cotangent $\lambda_{k+1}$.
- It is a matrix-vector product, so $\Phi'_k$ is never formed, let alone the full-system Jacobian.

Every operation supplies a local derivative rule, and AD chains those local rules into the whole-model gradient. We name these local operations in §4.

::: {.callout-note}
## Block back-substitution is the scalar algorithm, blockwise
A scalar upper-triangular solve $Ux=b$ runs bottom-up: $x_k = \big(b_k - \sum_{j>k} U_{kj}\,x_j\big)/U_{kk}$. A *block* triangular system uses the identical algorithm, just promote scalars to blocks: "multiply by $U_{kj}$" becomes a matrix-vector product, and "divide by $U_{kk}$" becomes "apply $U_{kk}^{-1}$", i.e. solve the small system $U_{kk}\,x_k=\text{rhs}$, never forming the inverse.

- *Explicit step*: diagonal blocks are $I$, so the divide is trivial: pure substitution, exactly the recurrence above.
- *Implicit step* (e.g. backward-Euler): the diagonal block is the step's own Jacobian, so each "divide" is one small linear solve.

Either way, it is the discrete adjoint of the solver actually being differentiated.
:::


### The gradient, accumulated

The backward sweep gives the big $\boldsymbol{\lambda}$. Feed it into §2's gradient formula:

$$\frac{dL}{d\theta}=\frac{\partial L}{\partial\theta}-\boldsymbol{\lambda}^{\top}\frac{\partial g}{\partial\theta}.$$

Two things collapse this to a clean per-step sum:

- $\partial L/\partial\theta = 0$ here: the loss sees $\theta$ *only through the states*, never directly. (It would reappear as $+\,\partial L/\partial\theta$ if the loss had an explicit term, such as a regularizer $R(\theta)$.)
- $\partial g/\partial\theta$ is stacked over the constraints, and every block vanishes except where $\theta$ enters that step:

$$\frac{\partial g}{\partial\theta} =
\begin{pmatrix}
\mathbf{0}\\
-\,\partial\Phi(u_0,\theta)/\partial\theta\\
\vdots\\
-\,\partial\Phi(u_{N-1},\theta)/\partial\theta
\end{pmatrix}.$$

So the gradient is just the block inner product of the big $\boldsymbol{\lambda}$ we solved for with this stacked $\partial g/\partial\theta$, which telescopes to one term per step:

$$\frac{dL}{d\theta} = -\,\boldsymbol{\lambda}^{\top}\frac{\partial g}{\partial\theta}
= \sum_{k=0}^{N-1}\lambda_{k+1}^{\top}\,\frac{\partial\Phi(u_k,\theta)}{\partial\theta}.$$

That sum is the accumulation:

- *Per-step* parameters $\theta_k$: only one term is nonzero.
- *Shared* parameters (the UDE / NN case, the same network reused every step): every step contributes and they add. This is why backprop sums a weight's gradient over all the places it appears.

The demo builds $\boldsymbol{\lambda}$ backward, then sums it into the gradient, for a shared $\theta$:

In [3]:
# A few steps of a 2-D NONLINEAR system, ONE shared parameter theta used at EVERY step -- like a UDE.
d, N, dt = 2, 4, 0.4
M = jnp.array([[-1.0, 0.8], [-0.8, -1.0]])
theta = 0.9                                              # SHARED across all steps
u0, target = jnp.array([1.0, 0.5]), jnp.array([0.2, -0.1])

def step(u, th): return u + dt * th * (M @ jnp.tanh(u))  # Phi(u, theta), mild nonlinearity
def rollout(th):
    us = [u0]
    for _ in range(N): us.append(step(us[-1], th))
    return jnp.stack(us)
def loss(th): return jnp.sum((rollout(th)[-1] - target) ** 2)

us = rollout(theta)
# Phi'_k = d step / d u_k = I + dt*theta * M @ diag(1 - tanh(u_k)^2)   (varies per step now)
Pp = [jnp.eye(d) + dt * theta * (M @ jnp.diag(1 - jnp.tanh(us[k]) ** 2)) for k in range(N)]

# (1) the BIG lambda: full transposed solve, and the backward recurrence -- identical
n = (N + 1) * d
G = jnp.eye(n)
for k in range(N): G = G.at[(k+1)*d:(k+2)*d, k*d:(k+1)*d].set(-Pp[k])   # block lower-bidiagonal
dLdU    = jnp.zeros(n).at[N*d:].set(2.0 * (us[-1] - target))            # loss touches only u_N
lam_big = jnp.linalg.solve(G.T, dLdU).reshape(N + 1, d)
lam = [None] * (N + 1); lam[N] = 2.0 * (us[-1] - target)
for k in range(N - 1, -1, -1): lam[k] = Pp[k].T @ lam[k + 1]
print("the big lambda, built BACKWARD (lambda_N -> lambda_0):")
for k in range(N, -1, -1): print(f"   lambda_{k} = {np.round(np.array(lam[k]), 4)}")
print("   full-system solve == backward recurrence:", bool(jnp.allclose(lam_big, jnp.stack(lam))))

# (2) gradient = SUM over steps of lambda_{k+1} . dPhi/dtheta,  with dPhi/dtheta = dt * (M @ tanh(u_k))
print("\ndL/dtheta accumulates one contribution per step (theta is shared):")
run = 0.0
for k in range(N):
    c = float(lam_big[k + 1] @ (dt * (M @ jnp.tanh(us[k]))))
    run += c
    print(f"   + step {k}:  lambda_{k+1} . dPhi/dtheta = {c:+.5f}    (running sum {run:+.5f})")
print(f"   = total {run:+.6f}     jax.grad = {float(jax.grad(loss)(theta)):+.6f}")

the big lambda, built BACKWARD (lambda_N -> lambda_0):
   lambda_4 = [-0.01   -0.3358]
   lambda_3 = [ 0.0757 -0.2246]
   lambda_2 = [ 0.1012 -0.1235]
   lambda_1 = [ 0.1007 -0.0509]
   lambda_0 = [ 0.0917 -0.0137]
   full-system solve == backward recurrence: True

dL/dtheta accumulates one contribution per step (theta is shared):
   + step 0:  lambda_1 . dPhi/dtheta = +0.00601    (running sum +0.00601)
   + step 1:  lambda_2 . dPhi/dtheta = +0.00866    (running sum +0.01467)
   + step 2:  lambda_3 . dPhi/dtheta = +0.00910    (running sum +0.02377)
   + step 3:  lambda_4 . dPhi/dtheta = +0.01103    (running sum +0.03480)


   = total +0.034802     jax.grad = +0.034802


### Example B: a neural network is the same problem

The same story, with "time step" → "layer." A feedforward network is a composition,

$$x_0 = \text{input},\qquad x_{k+1} = f_k(x_k,\theta_k),\qquad L = \ell(x_n),\qquad x_k\in\mathbb{R}^{d_k}.$$

Treat each layer's output as an unknown and each layer as a constraint $g_{k+1} = x_{k+1}-f_k(x_k,\theta_k)=0$. The adjoint of the *whole network* is again one big transposed solve. For a plain chain of layers, $\partial g/\partial x$ is block lower-bidiagonal, with $f'_k\equiv\partial f_k/\partial x_k$:

$$\frac{\partial g}{\partial x}=
\begin{pmatrix}
I & & & \\
-f'_0 & I & & \\
& \ddots & \ddots & \\
& & -f'_{n-1} & I
\end{pmatrix},
\qquad
\Big(\frac{\partial g}{\partial x}\Big)^{\!\top}=
\begin{pmatrix}
I & -f'^{\top}_0 & & \\
& I & \ddots & \\
& & \ddots & -f'^{\top}_{n-1} \\
& & & I
\end{pmatrix}.$$

The structure decomposes that big solve into a backward sweep:

- Diagonal blocks are identities, so no inverse is needed: pure back-substitution.
- Off-diagonal blocks $f'_k\in\mathbb{R}^{d_{k+1}\times d_k}$ may be rectangular when layers change width; the triangular structure remains.
- The backward sweep is exactly *backpropagation*:

$$\lambda_n = \Big(\frac{\partial\ell}{\partial x_n}\Big)^{\!\top},\qquad
\lambda_k = \Big(\frac{\partial f_k}{\partial x_k}\Big)^{\!\top}\lambda_{k+1},\qquad
\frac{\partial L}{\partial\theta_k} = \Big(\frac{\partial f_k}{\partial\theta_k}\Big)^{\!\top}\lambda_{k+1}.$$

If the loss also touches intermediate activations, the local source term simply reappears:

$$\lambda_k =
\Big(\frac{\partial L}{\partial x_k}\Big)^{\!\top}
+
\Big(\frac{\partial f_k}{\partial x_k}\Big)^{\!\top}\lambda_{k+1}.$$

- More complex networks fit the same framework:

>  A chain gives a block lower-bidiagonal constraint Jacobian. A general acyclic computation graph gives a **sparse block lower-triangular** Jacobian after a topological ordering. Backprop is then reverse topological substitution: each node sends transpose-Jacobian contributions to its parents, and contributions from multiple children add. Skip connections, branches, merges, and shared weights change the sparsity pattern, not the principle.

For example, a residual block

$$x_{k+1}=x_k+F_k(x_k,\theta_k)$$

has the backward update

$$\lambda_k = \lambda_{k+1} + \Big(\frac{\partial F_k}{\partial x_k}\Big)^{\!\top}\lambda_{k+1},$$

so the skip connection contributes the identity path in the adjoint.

Same three reasons the adjoint is the right tool as for a glacier: scalar loss, many parameters, and a triangular $\partial g/\partial x$ that turns the big solve into a cheap backward pass. For an MLP $f_k(x)=\sigma(W_k x+b_k)$ it is the textbook rule $\lambda_k=W_k^{\!\top}\big(\sigma'(z_k)\odot\lambda_{k+1}\big)$, matching `jax.grad`:


In [4]:
import jax.random as jr

# A small MLP: tanh hidden layers, linear output.  Params = list of (W, b).
sizes = [4, 16, 16, 1]
keys  = jr.split(jr.PRNGKey(0), len(sizes) - 1)
params = [(0.6 * jr.normal(k, (n_out, n_in)), jnp.zeros(n_out))
          for k, n_in, n_out in zip(keys, sizes[:-1], sizes[1:])]
x_in, y_tgt = jnp.array([0.3, -0.7, 1.1, 0.2]), jnp.array([0.5])

def mlp_loss(params, x):
    a = x
    for W, b in params[:-1]:
        a = jnp.tanh(W @ a + b)          # hidden layers
    W, b = params[-1]
    return jnp.sum((W @ a + b - y_tgt) ** 2)   # linear output + squared error

g_ad = jax.grad(mlp_loss)(params, x_in)        # reverse-mode AD

def hand_backprop(params, x):
    # forward pass, storing the states the backward pass will need
    a, acts, zs = x, [x], []
    for W, b in params[:-1]:
        z = W @ a + b; a = jnp.tanh(z); zs.append(z); acts.append(a)
    W, b = params[-1]; out = W @ a + b
    # backward pass = the adjoint recurrence  lambda_l = (dPhi_l/da_l)^T lambda_{l+1}
    lam = 2.0 * (out - y_tgt)                   # lambda = dL/d(output)
    grads = [(jnp.outer(lam, acts[-1]), lam)]   # output-layer gradients
    lam = params[-1][0].T @ lam                 # push cotangent into last hidden activation
    for l in range(len(params) - 2, -1, -1):
        delta   = (1.0 - jnp.tanh(zs[l]) ** 2) * lam      # through the tanh
        grads.insert(0, (jnp.outer(delta, acts[l]), delta))
        lam = params[l][0].T @ delta                      # (dPhi/da)^T = W^T (.)
    return grads

g_hand = hand_backprop(params, x_in)
worst = max(float(jnp.max(jnp.abs(a - b)))
            for (gw, gb), (hw, hb) in zip(g_ad, g_hand) for a, b in [(gw, hw), (gb, hb)])
print(f"max |AD - hand-coded backprop| over all weights & biases = {worst:.2e}")

max |AD - hand-coded backprop| over all weights & biases = 4.44e-16


::: {.callout-important}
## Takeaway
A glacier solver and a neural network look different, but their scalar-loss gradients have the same discrete-adjoint structure:

- Write the model as constraints $g=0$.
- Solve the transposed adjoint system $(\partial g/\partial u)^{\!\top}\lambda=(\partial L/\partial u)^{\!\top}$.
- Accumulate parameter gradients from local transpose-Jacobian products.

Reverse-mode AD automates this: each operation supplies its local derivative rule, and AD chains those rules into the whole-model gradient.
:::


## 4 · Forward mode: the same system, the other direction

Reverse mode gave the whole gradient in one backward pass. Forward mode is the other way to use the same system, and it wins in the opposite regime. Which mode to reach for is set by the shape of the Jacobian:

- *Scalar loss, many parameters* (a $1\times n$ Jacobian): reverse is usually the right default. One backward pass yields the entire gradient, whereas forward would need one pass per input direction. This is the setting for training and inversion.
- *Few inputs, many outputs* (an $m\times 1$ Jacobian): forward wins. One forward pass suffices, whereas reverse would need one pass per output direction. A typical case is the sensitivity of a whole trajectory to a single parameter.
- *Memory*: forward carries tangents alongside the values, so it needs no history ($O(1)$ extra memory); reverse must revisit each forward state on the way back, so it stores or recomputes them (the reason long solves need checkpointing, §5).

The two modes also compute *different objects*, which is worth stating plainly:

- Reverse propagates the adjoint $\boldsymbol{\lambda}$ (how the loss responds to each state), and returns $dL/d\theta$ for every parameter at once.
- Forward propagates the sensitivity $d\mathbf{U}/d\theta$ (how every state responds to one chosen parameter direction). This is a *directional* derivative; contracting it with $\partial L/\partial \mathbf{U}$ gives the loss's slope along that single direction, and you repeat over directions if you need the full gradient.

Mechanically, forward solves the *same* big system as Example A, as written rather than transposed. Differentiating $g(\mathbf{U},\theta)=0$ gives one big linear solve:

$$\Big(\frac{\partial g}{\partial \mathbf{U}}\Big)\frac{d\mathbf{U}}{d\theta} = -\frac{\partial g}{\partial\theta}.$$

Because $\partial g/\partial \mathbf{U}$ is the same block lower-bidiagonal matrix, forward-substitution (top down) decomposes it into a per-step *tangent* recurrence:

$$\frac{dx_0}{d\theta} = 0,\qquad \frac{dx_{k+1}}{d\theta} = \Phi'_k\,\frac{dx_k}{d\theta} + \frac{\partial\Phi(x_k,\theta)}{\partial\theta}.$$

Here $\Phi'_k\,(dx_k/d\theta)$ applies the step's Jacobian to the incoming tangent: again a matrix-vector product, but front to back this time.

Reverse solves the transpose bottom up (a cotangent, pulled back); forward solves the original system top down (a tangent, pushed forward). Same triangular matrix, opposite ends.

::: {.callout-note}
## Now name them: JVP and VJP
Look back at both recurrences. Each does one thing per step: apply a local Jacobian, or its transpose, to a vector.

- Forward applies $\Phi'_k$ to a tangent, $t\mapsto\Phi'_k\,t$. This is a **Jacobian-vector product** (JVP, `jax.jvp`): one pass per input direction.
- Reverse applies $\Phi'^{\top}_k$ to a cotangent, $\bar a\mapsto\Phi'^{\top}_k\,\bar a$. This is a **vector-Jacobian product** (VJP, `jax.vjp`): one pass per output direction. A scalar-loss gradient is the VJP seeded with $1$.

So the recurrences never form $\Phi'_k$, let alone the full-system Jacobian: each operation only supplies these matrix-vector rules, and AD chains them.
:::

The tangent recurrence, checked against the full forward solve and `jax.jvp` on the same 2-D system as Example A:


In [5]:
# Forward mode on Example A's 2-D system (reusing us, Pp, G, M, dt, N, d, theta, target, loss).
nU = (N + 1) * d

# (1) ONE big solve, as written:  (dg/dU) dU/dtheta = -dg/dtheta
dg_dtheta = np.zeros(nU)
for k in range(N):
    dg_dtheta[(k+1)*d:(k+2)*d] = -np.asarray(dt * (M @ jnp.tanh(us[k])))     # dg_{k+1}/dtheta
dUdtheta_full = np.linalg.solve(np.asarray(G), -dg_dtheta)

# (2) the SAME solve as a per-step tangent recurrence (forward-substitution, top-down)
s = [np.zeros(d)]
for k in range(N):
    s.append(np.asarray(Pp[k]) @ s[-1] + np.asarray(dt * (M @ jnp.tanh(us[k]))))
s = np.concatenate(s)
print("forward-substitution == full forward solve:", bool(np.allclose(dUdtheta_full, s)))

# forward gradient: one input (theta) -> a single JVP.  dL/dtheta = dL/du_N . du_N/dtheta
dL_duN = 2.0 * (np.asarray(us[-1]) - np.asarray(target))
print(f"dL/dtheta (forward-substitution) = {float(dL_duN @ s[-d:]):+.6f}")
print(f"dL/dtheta (jax.jvp)              = {float(jax.jvp(loss, (theta,), (1.0,))[1]):+.6f}")

forward-substitution == full forward solve: True
dL/dtheta (forward-substitution) = +0.034802


dL/dtheta (jax.jvp)              = +0.034802


::: {.callout-note}
## AD vs. the adjoint method
- *AD* supplies local derivative rules and a way to compose them through a program: forward mode uses JVPs, reverse mode uses VJPs.
- *The adjoint method* is the reverse sensitivity calculation written as a transposed solve with adjoint variables. It predates modern AD and can be derived by hand.
- In the scalar-loss, many-parameter setting, reverse-mode AD is the adjoint method automated. Forward mode is the complementary tool: less efficient for full scalar-loss gradients with many parameters, but ideal for directional derivatives and small-input sensitivity studies.
:::


## 5 · Checkpointing: differentiating through long solves

::: {.callout-note}
## Common misconception
By default, neither JAX nor PyTorch checkpoints. The default reverse pass is maximum-memory, zero-recompute: save every intermediate, consume it in reverse.

- *PyTorch (eager):* saves activations; opt-in recompute is `torch.utils.checkpoint`.
- *JAX:* `jax.grad` saves residuals; opt-in rematerialization is `jax.checkpoint` (`jax.remat`).
- *Caveats:* XLA can rematerialize to fit a memory budget (heuristic), and `torch.compile`'s min-cut partitioner makes automatic save-vs-recompute choices; neither is the principled $O(\log N)$ schedule.
:::

Checkpointing trades memory for recomputation: store a few states, recompute the rest during the backward pass. The optimal schedule (Griewank and Walther's `revolve`) is $O(\log N)$ memory at $O(N\log N)$ compute.

::: {.callout-important}
## Why this matters for time-dependent inversions and UDEs
Training a UDE backpropagates through every time step. Look at where the forward state enters the reverse recurrence:

$$\lambda_k = \frac{\partial L}{\partial u_k} + \underbrace{\Big(\frac{\partial\Phi(u_k,\theta)}{\partial u_k}\Big)^{\!\top}}_{\text{needs the forward state }u_k}\lambda_{k+1}.$$

Every step needs its $u_k$ to evaluate $\Phi'_k$, so the forward trajectory must be available on the way back. Storing all of it is $O(N)$ in the number of steps:

- our toy glacier: ~600 steps for a spin-up, so storing all is fine;
- a real inversion (Bolibar et al.: ~$10^5$ ODEs over a 2-D grid, many years): storing every state is infeasible.

Checkpointing keeps only selected states and recomputes the rest forward, which is safe because the forward solve only ever runs forward in time. This is why the lecture's solver passes `RecursiveCheckpointAdjoint`.
:::

Hand-rolled `revolve` on a chain of $N$ time steps, to see the memory/compute trade and confirm the gradient is unchanged:


In [6]:
N, theta = 64, 0.99
def phi(y):  return np.tanh(y) + theta * y
def dphi(y): return (1 - np.tanh(y) ** 2) + theta

def store_all(y0):
    ys = [y0]
    for _ in range(N): ys.append(phi(ys[-1]))
    lam = ys[N]
    for k in range(N - 1, -1, -1): lam = dphi(ys[k]) * lam
    return lam, len(ys), N

def checkpointed(y0, s):
    ckpts, y = {0: y0}, y0
    for k in range(1, N + 1):
        y = phi(y)
        if k % s == 0: ckpts[k] = y
    lam, fwd, peak = y, N, max(len(ckpts), s)
    for k in range(N - 1, -1, -1):
        if k in ckpts:
            yk = ckpts[k]
        else:
            base = max(c for c in ckpts if c < k); yk = ckpts[base]
            for _ in range(k - base): yk = phi(yk); fwd += 1
        lam = dphi(yk) * lam
    return lam, peak, fwd

g0, m0, f0 = store_all(1.0)
g1, m1, f1 = checkpointed(1.0, s=8)
print(f"store-all    : ~{m0:2d} states, {f0:3d} forward evals")
print(f"checkpointed : ~{m1:2d} states, {f1:3d} forward evals   (same gradient: {abs(g0-g1) < 1e-9})")

def deep(x):
    for _ in range(50): x = jnp.tanh(x) + 0.1 * x
    return jnp.sum(x ** 2)
x0 = jnp.linspace(-1, 1, 8)
print("jax.grad == jax.grad(jax.checkpoint(...)):", bool(jnp.allclose(jax.grad(deep)(x0), jax.grad(jax.checkpoint(deep))(x0))))

store-all    : ~65 states,  64 forward evals
checkpointed : ~ 9 states, 288 forward evals   (same gradient: True)


jax.grad == jax.grad(jax.checkpoint(...)): True


### References

- MacAyeal (1993), *Control methods in ice-sheet modeling*, J. Glaciol. 39(131).
- Griewank & Walther (2008), *Evaluating Derivatives* (SIAM); Griewank (1992), `revolve`, Optim. Methods Softw. 1.
- Baydin et al. (2018), *AD in machine learning: a survey*, JMLR 18(153).
- Farrell, Ham, Funke & Rognes (2013), *Automated derivation of the adjoint of high-level FE programs*, SIAM J. Sci. Comput. 35(4); Rathgeber et al. (2016), *Firedrake*, ACM TOMS 43(3); Alnæs et al. (2014), *UFL*, ACM TOMS 40(2).
- Chen et al. (2018), *Neural ODEs*, NeurIPS; Kidger (2021), *On Neural Differential Equations* (thesis; `diffrax`).
- Sapienza et al. (2024), *Differentiable programming for differential equations: a review*, arXiv:2406.09699.

---
*Companion to the UDE glaciology lecture.*